# 🕹️ Deep RL Evolution Pipeline — REINFORCE → A2C → PPO

**Environment:** `ALE/Pong-v5` from raw pixels  
**Runtime:** GPU (T4 recommended) | Runtime → Change runtime type → GPU

---

## 1 · Environment Setup
Install all dependencies. This cell only needs to run once per Colab session.

In [ ]:
# ── System packages ──────────────────────────────────────────────────────────
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1

# ── Python packages ──────────────────────────────────────────────────────────
!pip install -q \
    gymnasium==0.29.1 \
    'gymnasium[atari]' \
    'gymnasium[accept-rom-license]' \
    opencv-python-headless \
    pyvirtualdisplay \
    torch torchvision \
    ale-py \
    imageio imageio-ffmpeg \
    matplotlib pandas

print('✅ All packages installed.')

In [ ]:
import os, sys

# ── Clone / pull the project repo ────────────────────────────────────────────
REPO_URL  = 'https://github.com/YOUR_USERNAME/rl-atari-evolution.git'  # ← update
REPO_NAME = 'rl-atari-evolution'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    !git -C {REPO_NAME} pull

# Put the project root on the Python path
ROOT = os.path.abspath(REPO_NAME)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print(f'📂 Working directory: {os.getcwd()}')

## 2 · Virtual Display
Colab has no physical display — we start Xvfb so ALE can render frames.

In [ ]:
from visualization.display import init_virtual_display, VirtualDisplayContext

# Start the virtual framebuffer (1400×900)
_display = init_virtual_display()
print('Virtual display ready ✓')

## 3 · Sanity Check — Wrapped Environment
Verify wrapper chain, observation shape, and action space before training.

In [ ]:
import numpy as np
from config import CFG
from src.wrappers import make_atari_env

env = make_atari_env(CFG.ENV_NAME, seed=CFG.SEED)
obs, info = env.reset(seed=CFG.SEED)
obs_arr = np.asarray(obs)

print(f'Environment : {CFG.ENV_NAME}')
print(f'Obs shape   : {obs_arr.shape}  (expect (4, 84, 84))')
print(f'Obs dtype   : {obs_arr.dtype}  (expect float32)')
print(f'Obs range   : [{obs_arr.min():.3f}, {obs_arr.max():.3f}]  (expect [0, 1])')
print(f'Action space: {env.action_space}  (6 discrete actions)')
env.close()

## 4 · REINFORCE Training

Adjust `--episodes` to taste.  
- **Quick smoke-test:** 50 episodes (~2 min on GPU)  
- **Meaningful learning:** 2 000–5 000 episodes (may take 1–4 h on T4)

Progress is printed every 10 episodes and logged to `logs/reinforce_rewards.csv`.

In [ ]:
# ── Training — edit flags as needed ──────────────────────────────────────────
!python -m src.train \
    --algo reinforce \
    --episodes 1000 \
    --save-freq 50 \
    --log-interval 10 \
    --seed 42

## 5 · Live Reward Plot
Plot the moving average from the CSV log to track learning progress.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = 'logs/reinforce_rewards.csv'
df = pd.read_csv(log_path)

WINDOW = 50
df['moving_avg'] = df['reward'].rolling(WINDOW, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Episode rewards
axes[0].plot(df['episode'], df['reward'], alpha=0.3, color='steelblue', label='Episode reward')
axes[0].plot(df['episode'], df['moving_avg'], color='navy', linewidth=2,
             label=f'{WINDOW}-ep moving avg')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title('REINFORCE — Training Rewards')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Policy loss
axes[1].plot(df['episode'], df['loss'], alpha=0.5, color='tomato', label='Policy loss')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Loss')
axes[1].set_title('REINFORCE — Policy Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('logs/reinforce_training_curve.png', dpi=120)
plt.show()
print('Plot saved to logs/reinforce_training_curve.png')

## 6 · Visual Evaluation — Gameplay Video

Load the latest checkpoint, run the agent for 20 seconds (≈ 1 episode), record an `.mp4`, and display it inline.

In [ ]:
import glob, os

checkpoints = sorted(
    glob.glob('checkpoints/reinforce/ep_*.pt'),
    key=lambda p: int(os.path.basename(p).replace('ep_', '').replace('.pt', ''))
)

if not checkpoints:
    # Fall back to the final checkpoint if individual episode ones don't exist yet
    checkpoints = glob.glob('checkpoints/reinforce/final.pt')

if not checkpoints:
    print('⚠️  No checkpoint found. Run the training cell first.')
else:
    latest_ckpt = checkpoints[-1]
    print(f'Latest checkpoint: {latest_ckpt}')

In [ ]:
import torch
import numpy as np
import gymnasium as gym
from IPython.display import Video, display as ipy_display

from config import CFG
from src.wrappers import make_eval_env
from src.reinforce import REINFORCEAgent

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
VIDEO_DIR = 'videos/reinforce'
FPS       = 30
MAX_STEPS = 20 * FPS   # 20-second clip at 30 fps

# ── Load agent ────────────────────────────────────────────────────────────────
eval_env  = make_eval_env(CFG.ENV_NAME, seed=0, render_mode='rgb_array')
action_dim = eval_env.action_space.n

agent = REINFORCEAgent(action_dim=action_dim, device=DEVICE)
agent.load(latest_ckpt)
agent.policy.eval()

# ── Wrap with RecordVideo ─────────────────────────────────────────────────────
import os; os.makedirs(VIDEO_DIR, exist_ok=True)

rec_env = gym.wrappers.RecordVideo(
    eval_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda ep: ep == 0,   # record only 1 episode
    name_prefix='reinforce_eval',
    video_length=MAX_STEPS,
)

obs, _ = rec_env.reset(seed=0)
done = False
total_reward = 0.0
step = 0

while not done and step < MAX_STEPS:
    obs_t = torch.tensor(
        np.asarray(obs, dtype=np.float32), dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        action, _ = agent.policy.get_action(obs_t)

    obs, reward, terminated, truncated, _ = rec_env.step(action)
    total_reward += float(reward)
    done = terminated or truncated
    step += 1

rec_env.close()
print(f'Episode complete — steps: {step}, total reward: {total_reward:.1f}')

# ── Find and display the saved video ─────────────────────────────────────────
import glob
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
if videos:
    video_path = videos[-1]
    print(f'Displaying: {video_path}')
    ipy_display(Video(video_path, embed=True, width=420))
else:
    print('⚠️  No video file found. Check that ffmpeg is installed (apt-get install ffmpeg).')

## 7 · Next Steps

| Step | Command |
|---|---|
| Train A2C | `!python -m src.train --algo a2c --episodes 3000` |
| Train PPO | `!python -m src.train --algo ppo --episodes 5000` |
| Compare all three | Run `benchmarks/evaluator.py` |

---
*Remember: REINFORCE needs many more episodes than A2C/PPO to learn Pong. 
Scores above -10 after 2000 episodes indicate the agent is learning.*